# ✏️ Práctica para alumnos
## Pre-entrega: Adquisición y Normalización con Pandas — Librería "Página Nueva"

Vas a resolver la misma consigna que vimos en clase (ElectroSur), pero con un dataset distinto que tiene **problemas adicionales a propósito**, incluyendo el error común que menciona la consigna: *"Intentar hacer un merge con columnas de tipos diferentes (ej. ID como string en uno y entero en otro)"*.

**Escenario:** La librería "Página Nueva" registra sus ventas en un sistema que exporta a **CSV**, y mantiene el catálogo de libros en una planilla de **Excel** cargada manualmente por el área de compras (por eso tiene más inconsistencias).

**Archivos de este ejercicio** (deben estar en la misma carpeta que este notebook):
- `transacciones_libreria.csv` → `id_transaccion`, `id_libro`, `cantidad`, `fecha`
- `catalogo_libros.xlsx` → `id_libro`, `titulo`, `precio_unitario`, `categoria`

Completá los `TODO`. Cada sección tiene una celda de **auto-chequeo** (`assert`) al final — si no marca error, ¡vas bien! Si te trabás, repasá el notebook resuelto de la clase (`Clase_Adquisicion_y_Normalizacion_ElectroSur.ipynb`).

**Pasos** (igual que la consigna original):
1. Adquisición
2. Transformación (tipos de datos, nulos, duplicados, merge)
3. Normalización (`total_venta`)
4. Validación
5. Exportación a Parquet


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

RUTA_TRANSACCIONES = Path("transacciones_libreria.csv")
RUTA_CATALOGO = Path("catalogo_libros.xlsx")
RUTA_SALIDA = Path("ventas_libreria_consolidado.parquet")

print("Archivos disponibles:", RUTA_TRANSACCIONES.exists(), RUTA_CATALOGO.exists())


## 1. Adquisición

Cargá ambos archivos con la función de pandas correspondiente.

In [ ]:
# TODO: cargar el CSV de transacciones
transacciones = None

# TODO: cargar el Excel del catálogo (necesita openpyxl instalado)
catalogo = None

print(transacciones.shape, catalogo.shape)
transacciones.head()


In [ ]:
# ✅ Auto-chequeo
assert transacciones is not None and len(transacciones) > 0, "No se cargaron las transacciones"
assert catalogo is not None and len(catalogo) > 0, "No se cargó el catálogo"
assert list(catalogo.columns) == ["id_libro", "titulo", "precio_unitario", "categoria"]
print("Ejercicio 1 OK ✅")


## 2. Transformación

### 2.1 Exploración inicial

Antes de tocar nada: ¿cuántos nulos hay? ¿qué tipo de dato tiene cada columna clave?

In [ ]:
# TODO 1: mostrar transacciones.info()


# TODO 2: contar nulos por columna en 'transacciones' y guardarlo en `nulos_transacciones`
nulos_transacciones = None

# TODO 3: comparar el dtype de 'id_libro' en transacciones vs. catalogo (imprimí ambos)


print(nulos_transacciones)


In [ ]:
# ✅ Auto-chequeo
assert nulos_transacciones is not None
assert nulos_transacciones["cantidad"] > 0, "Se esperaban nulos en 'cantidad': revisá que hayas usado .isnull().sum()"
print("Ejercicio 2.1 OK ✅ — encontraste los nulos en 'cantidad'")


### 2.2 🔎 El error común: tipos de datos incompatibles en el ID

Vas a notar que `id_libro` en `transacciones` es de tipo **texto** (`object`), mientras que en `catalogo` es **numérico** (`int64`). Además, algunas filas de `transacciones` tienen valores de `id_libro` que **no son números** (por ejemplo `"N/D"`, `"?"`, cadenas vacías).

Si intentás el merge así nomás, **no vas a obtener ningún error, pero el resultado va a estar mal**: pandas comparará `"5"` (texto) contra `5` (número) y nunca van a coincidir, perdiendo silenciosamente todas las filas. Este es exactamente el error común que menciona la consigna.

**Tu tarea:**
1. Convertí los valores de `id_libro` que no son números a `NaN` (usá `pd.to_numeric` con `errors="coerce"`, que convierte lo inconvertible en `NaN` en vez de lanzar una excepción).
2. Eliminá las filas donde `id_libro` haya quedado nulo (no podemos adivinar a qué libro corresponde esa venta).
3. Convertí la columna a `int64` para que combine con `catalogo["id_libro"]`.


In [ ]:
# Pista: mirá algunos valores 'raros' antes de empezar
print(transacciones["id_libro"].unique()[:15])


In [ ]:
# TODO 1: convertir id_libro a numérico, forzando NaN en los valores no convertibles
transacciones["id_libro"] = None

# TODO 2: eliminar las filas donde id_libro quedó nulo


# TODO 3: convertir id_libro a int64


In [ ]:
# ✅ Auto-chequeo
assert transacciones["id_libro"].dtype == catalogo["id_libro"].dtype, \
    "Los dtypes de 'id_libro' todavía no coinciden entre ambos datasets"
assert transacciones["id_libro"].isnull().sum() == 0
print("Ejercicio 2.2 OK ✅ — resolviste el problema de tipos de datos")


### 2.3 Corregir el tipo de dato de `fecha`

Igual que en el ejemplo de clase: hay fechas en formato `dd/mm/yyyy` mezcladas con `yyyy-mm-dd`, y **una fecha inválida** (`31/02/2025`, ¡febrero no tiene 31 días!). Usá `pd.to_datetime(..., format="mixed", errors="coerce")` para que la fecha inválida se convierta en `NaT` en vez de romper todo el proceso, y luego decidí qué hacer con esa fila.

In [ ]:
# TODO 1: convertir 'fecha' a datetime con format="mixed" y errors="coerce"


# TODO 2: contar cuántas fechas quedaron como NaT (nulas) y guardarlo en `fechas_invalidas`
fechas_invalidas = None

# TODO 3: eliminar las filas con fecha inválida (NaT)


print("Fechas inválidas encontradas:", fechas_invalidas)


In [ ]:
# ✅ Auto-chequeo
assert pd.api.types.is_datetime64_any_dtype(transacciones["fecha"])
assert transacciones["fecha"].isnull().sum() == 0
assert fechas_invalidas is not None and fechas_invalidas >= 1
print("Ejercicio 2.3 OK ✅")


### 2.4 Nulos en `cantidad` y duplicados

Aplicá el mismo criterio que en clase: no se puede inventar una cantidad vendida, así que esas filas se eliminan. También eliminá filas duplicadas.

In [ ]:
# TODO 1: eliminar filas donde 'cantidad' sea nula, y convertir la columna a int


# TODO 2: eliminar filas duplicadas


print(f"Filas restantes: {len(transacciones)}")


In [ ]:
# ✅ Auto-chequeo
assert transacciones["cantidad"].isnull().sum() == 0
assert transacciones.duplicated().sum() == 0
assert transacciones["cantidad"].dtype.kind in "iu", "cantidad debería ser un entero"
print("Ejercicio 2.4 OK ✅")


### 2.5 Merge

Ahora que los tipos de dato coinciden, hacé el merge. Usá `how="left"` e `indicator=True` (como en el ejemplo de clase) para poder detectar transacciones de libros que no están en el catálogo (¡también los hay en este dataset!).

In [ ]:
# TODO 1: hacer el merge (how="left", indicator=True) sobre 'id_libro'
ventas = None

# TODO 2: filtrar y descartar las filas "left_only" (producto sin catálogo), y quedarte solo con las "both"
#         no te olvides de eliminar la columna auxiliar '_merge' y resetear el índice


ventas.head()


In [ ]:
# ✅ Auto-chequeo
assert ventas is not None
assert "_merge" not in ventas.columns, "No te olvidaste de eliminar la columna auxiliar '_merge'?"
assert ventas["titulo"].isnull().sum() == 0, "Todavía quedan filas sin libro asociado"
print("Ejercicio 2.5 OK ✅")


## 3. Normalización

Creá la columna `total_venta = cantidad * precio_unitario`.

In [ ]:
# TODO: crear la columna 'total_venta'


ventas[["id_transaccion", "titulo", "cantidad", "precio_unitario", "total_venta"]].head()


In [ ]:
# ✅ Auto-chequeo
assert "total_venta" in ventas.columns
assert (ventas["total_venta"] == ventas["cantidad"] * ventas["precio_unitario"]).all()
print("Ejercicio 3 OK ✅")


## 4. Validación final

Completá los chequeos de calidad antes de exportar (los mismos 4 que vimos en clase).

In [ ]:
# TODO: completar los 4 asserts de validación (nulos, duplicados, montos no negativos, fecha datetime)



print("Validaciones OK ✅")
print(f"Resumen final: {len(ventas)} transacciones por un total de ${ventas['total_venta'].sum():,.0f}")


## 5. Exportación a Parquet

In [ ]:
# TODO: exportar 'ventas' a RUTA_SALIDA en formato parquet


print(f"Archivo exportado: {RUTA_SALIDA} ({RUTA_SALIDA.stat().st_size / 1024:.1f} KB)" if RUTA_SALIDA.exists() else "Todavía no exportaste el archivo")


In [ ]:
# ✅ Auto-chequeo final
assert RUTA_SALIDA.exists(), "No se generó el archivo parquet"
verificacion = pd.read_parquet(RUTA_SALIDA)
assert len(verificacion) == len(ventas)
print("🎉 ¡Pre-entrega resuelta! El dataset consolidado está listo.")


## Reflexión final (para tu README.md)

Redactá 3-4 líneas explicando:
- Qué decisiones tomaste con los valores nulos y por qué.
- Qué hiciste con el problema de tipos de datos incompatibles en `id_libro`.
- Cuántas filas se perdieron en el proceso y por qué motivos (nulos, duplicados, IDs sin catálogo, fechas inválidas).

Esto es justamente lo que la consigna pide incluir en el `README.md` del repositorio — **no te olvides de subirlo**, es uno de los errores comunes que señala el enunciado.

_Tu respuesta:_
